In [1]:
import sys
import os
# Añadir el directorio raíz al path para poder importar src
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src import Extractor

TABLE_NAME = "works4cdp_assay"
SAMPLE_ID = 24

# Configuración de conexión idéntica a los notebooks de extracción
db_config = {
    'user': 'myuser', 
    'password': 'mypassword', 
    'host': 'localhost', 
    'port': '5433',
    'dbname': 'mydb'
}

# Inicializar extractor y obtener assays
extractor = Extractor(table_name=TABLE_NAME, **db_config)
df_assays = extractor.get_assays(SAMPLE_ID)

# Cargar el dataset clusterizado del notebook 03
cluster_path = 'data/processed/intensidad_cobre_24_clusterizado.csv'
df_cluster = pd.read_csv(cluster_path).dropna(subset=['instance'])

# 1. Convertir y limpiar date/time en df_assays
df_assays['date'] = df_assays['date'].astype(str).str.strip()
df_assays['time'] = df_assays['time'].astype(str).str.strip().apply(lambda x: x.split('.')[0] if '.' in x else x)

# 2. Seleccionar columnas procesadas del clustering
cols_cluster = [
    'date', 'time', 'instance', 
    'n1fe_ortho', 'n2cu_ortho', 'n3zn_ortho', 'n4mo_ortho', 
    'cluster_kmeans', 'cluster_gmm'
]
df_cluster_subset = df_cluster[cols_cluster].copy()

# 3. Convertir y limpiar date/time en df_cluster_subset
df_cluster_subset['date'] = df_cluster_subset['date'].astype(str).str.strip()
df_cluster_subset['time'] = df_cluster_subset['time'].astype(str).str.strip().apply(lambda x: x.split('.')[0] if '.' in x else x)

# 4. Fusionar mediante inner join
df_merged = pd.merge(df_assays, df_cluster_subset, on=['date', 'time'], how='inner')

# Guardar el CSV fusionado en la carpeta de processed
output_path = 'data/processed/intensidad_cobre_24_completo.csv'
df_merged.to_csv(output_path, index=False)



# Cargar los datos completos
#file_path = 'data/processed/intensidad_cobre_24_completo.csv'
#x = pd.read_csv(file_path)

col = ['pFe','pCu','pZn','pMo','pIns','pSol']
df_final = df_merged.dropna(subset=col, how='all')

ruta_salida = 'data/processed/intensidad_cobre_24_completo_filtrado.csv'
df_final.to_csv(ruta_salida, index=False)
